> `oeai_mod_arbor_env_var`
> [20251105.1]
> *Module configuration*

In [0]:
%run ./oeai_py

In [0]:
%run ./oeai_logger

In [0]:
# Create an instance of OEAI class and set the plaform ("Azure", "Fabric"...)
oeai = OEAI(platform="Fabric")
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [0]:
# CHANGE VALUES FOR YOUR KEY VAULT
keyvault = "" # Fabric requires full URL eg "https://key_vault_name.vault.azure.net/"
keyvault_linked_service = "" # Not required for Fabric.

In [0]:
# INITIALISE LOGGING
oeai.log = OEAILogger()
oeai.module_id = "arbor"

# SET HANDLER - Console Output
oeai.log.handler.stream = SimpleNamespace(
    level=_lib_logging.INFO, 
    formatter="stream"
)

In [0]:
# OEA environment paths
import os
from types import SimpleNamespace
storage_root = oeai.get_secret(spark, "storage-root", keyvault_linked_service, keyvault)
oeai.path = SimpleNamespace(
    reference = os.path.join(storage_root, f"reference/"),
    bronze    = os.path.join(storage_root, f"oeai_bronze/{oeai.module_id}/"),
    silver    = os.path.join(storage_root, f"oeai_silver/"),
    gold      = os.path.join(storage_root, f"oeai_gold/"),
)
bronze_path = oeai.path.bronze
silver_path = oeai.path.silver
gold_path   = oeai.path.gold

In [0]:
# Arbor: Connection details
arbor_db_Schema = oeai.get_secret(spark, "arbor-db-schema", keyvault_linked_service, keyvault)
arbor_db_Database = oeai.get_secret(spark, "arbor-db-database", keyvault_linked_service, keyvault)
arbor_db_Warehouse = oeai.get_secret(spark, "arbor-db-warehouse", keyvault_linked_service, keyvault)
arbor_db_Account = oeai.get_secret(spark, "arbor-db-account", keyvault_linked_service, keyvault)
arbor_db_Pass = oeai.get_secret(spark, "arbor-db-password", keyvault_linked_service, keyvault)
arbor_db_User = oeai.get_secret(spark, "arbor-db-user", keyvault_linked_service, keyvault)

In [0]:
from datetime import datetime

def get_ac_year(date=None):
    """
    Return the academic year for the given date.
    
    Academic year is the calendar year if month >= August, else the previous year.
    
    Args:
        date (str | datetime.datetime | None): 
            - If None, uses today.
            - If str, parsed with dateutil.parser.
            - If datetime, used directly.
    
    Returns:
        int: Academic year.
    
    Raises:
        ValueError: if string cannot be parsed as a date.
        TypeError: if date is not str or datetime.
    """
    # from datetime import datetime
    try:
        from dateutil import parser
    except ImportError:
        raise ImportError(
            "Please install python-dateutil to parse date strings: pip install python-dateutil"
        )

    # default to now
    if date is None:
        dt = datetime.now()
    # parse strings
    elif isinstance(date, str):
        try:
            dt = parser.parse(date, dayfirst=True)
        except (ValueError, TypeError) as e:
            raise ValueError(f"Could not parse date string {date!r}: {e}")
    # already datetime?
    elif isinstance(date, datetime):
        dt = date
    else:
        raise TypeError(f"date must be str or datetime, got {type(date).__name__!r}")

    return dt.year if dt.month >= 8 else dt.year - 1

In [0]:
### BRONZE ###
# List of queries for each view along with the view name
queries = [
    ("SELECT * FROM STUDENTS", "STUDENTS"),
    ("SELECT * FROM STUDENT_SEN_NEEDS", "STUDENT_SEN_NEEDS"),
    ("SELECT * FROM STUDENT_SCHOOL_ENROLMENTS" , "STUDENT_SCHOOL_ENROLMENTS"),
    ("SELECT * FROM STUDENT_REGISTRATION_FORM_MEMBERSHIPS" , "STUDENT_REGISTRATION_FORM_MEMBERSHIPS"),
    ("SELECT * FROM STUDENT_SEN_STATUS_ASSIGNMENT" , "STUDENT_SEN_STATUS_ASSIGNMENT"),
    ("SELECT * FROM STUDENT_TAGS_HISTORY" , "STUDENT_TAGS_HISTORY"),
    ("SELECT * FROM STUDENT_YEAR_GROUP_MEMBERSHIPS" , "STUDENT_YEAR_GROUP_MEMBERSHIPS"),
    ("SELECT * FROM YEAR_GROUPS" , "YEAR_GROUPS"),
    ("SELECT * FROM SUSPENSIONS" , "SUSPENSIONS"),
    ("SELECT * FROM SCHOOLS" , "SCHOOLS"),
    ("SELECT * FROM PERMANENT_EXCLUSIONS" , "PERMANENT_EXCLUSIONS"),
    ("SELECT * FROM STUDENT_ACADEMIC_YEAR_ENROLMENTS" , "STUDENT_ACADEMIC_YEAR_ENROLMENTS"),
    ("SELECT * FROM ROLL_CALL_ATTENDANCE WHERE DATE >= '{start_date}' AND DATE <= '{end_date}'", "ROLL_CALL_ATTENDANCE"),
    ("SELECT * FROM USER_DEFINED_FIELDS" , "USER_DEFINED_FIELDS"),
    ("SELECT * FROM USER_DEFINED_FIELDS_VALUES" , "USER_DEFINED_FIELDS_VALUES"),
    ("SELECT * FROM INTERNAL_EXCLUSIONS" , "INTERNAL_EXCLUSIONS"),
    ("SELECT * FROM BEHAVIOURAL_INCIDENTS" , "BEHAVIOURAL_INCIDENTS"),
    ("SELECT * FROM BEHAVIOURAL_INCIDENT_STUDENT_INVOLVEMENTS" , "BEHAVIOURAL_INCIDENT_STUDENT_INVOLVEMENTS"),
    ("SELECT * FROM REGISTRATION_FORMS" , "REGISTRATION_FORMS"),
    ("SELECT * FROM EXAM_RESULTS", "EXAM_RESULTS"),
    ("SELECT * FROM SUMMATIVE_ASSESSMENT_MARKS", "SUMMATIVE_ASSESSMENT_MARKS"),
    ("SELECT * FROM STANDARDISED_ASSESSMENT_MARKS", "STANDARDISED_ASSESSMENT_MARKS"),
    ("SELECT * FROM STAFF", "STAFF"),
    ("SELECT * FROM STAFF_ABSENCES", "STAFF_ABSENCES"),
    ("SELECT * FROM COURSES" , "COURSES"),
    ("SELECT * FROM COURSE_ENROLMENTS" , "COURSE_ENROLMENTS"),
    ("SELECT * FROM COURSE_LEADS" , "COURSE_LEADS"),
    ("SELECT * FROM LESSON_ATTENDANCE WHERE START_DATE_TIME >= '{start_date}' AND START_DATE_TIME <= '{end_date}'" , "LESSON_ATTENDANCE"),
    ("SELECT * FROM STAFF_CONTRACT", "STAFF_CONTRACT"),
    ("SELECT * FROM STUDENT_SCHOOL_ENROLMENTS", "STUDENT_SCHOOL_ENROLMENTS"),
    ("SELECT * FROM ACADEMIC_YEARS", "ACADEMIC_YEARS"),
    ("SELECT * FROM BEHAVIOURAL_INCIDENT_ACTIONS" , "BEHAVIOURAL_INCIDENT_ACTIONS"),
    ("SELECT * FROM DETENTIONS", "DETENTIONS"),
    ("SELECT * FROM INTERNAL_EXCLUSIONS", "INTERNAL_EXCLUSIONS"),
    ("SELECT * FROM POINT_AWARDS", "POINT_AWARDS"),
  ]

# Set the start date and the number of days for each batch
from datetime import datetime

# ——— CONFIGURATION ———
# If you want to hard-code an exact start date, set this to e.g. '2023-08-01'.
# Otherwise leave it as None to compute automatically.
override_start_date = '2025-08-01'  # e.g. '2023-08-01'

# ——— COMPUTE start_date ———
if override_start_date:
    # use the user-provided date
    start_date = datetime.strptime(override_start_date, '%Y-%m-%d')
else:
    # today = datetime.today()
    # year = today.year
    # # build this year’s August 1st
    # candidate = datetime(year, 8, 1)
    # # if today is before Aug 1, go back one year
    # if today < candidate:
    #     year -= 1
    start_date = datetime(get_ac_year(), 8, 1)
num_days = 30
print("Start date:", start_date.strftime('%Y-%m-%d'))

### SILVER ###

# List of tuples to hold the mappings
delta_table_name_mapping = [
    ("SCHOOLS", "dim_Organisation"),
    ("STUDENTS", "dim_Student"),
    ("STUDENTS", "dim_StudentExtended"),
    ("ROLL_CALL_ATTENDANCE", "fact_AttendanceSession"),
    ("SUSPENSIONS", "fact_Exclusion"),
    ("PERMANENT_EXCLUSIONS", "temp_fact_Exclusion"),
    ("STUDENT_ACADEMIC_YEAR_ENROLMENTS", "temp_dim_StudentEnrolments"),
    ("STUDENT_SCHOOL_ENROLMENTS", "temp_dim_StudentSchoolEnrolments"),
    ("STUDENT_ACADEMIC_YEAR_ENROLMENTS", "dim_StudentAcademicYearEnrolment"),
    ("STUDENT_SEN_STATUS_ASSIGNMENT", "temp_dim_StudentSENStatusAssignment"),
    ("STUDENT_TAGS_HISTORY", "dim_StudentExtendedHistory"),
    ("YEAR_GROUPS", "dim_Group"),
    ("STUDENT_YEAR_GROUP_MEMBERSHIPS", "dim_GroupMembership"),
    ("BEHAVIOURAL_INCIDENTS", "fact_Behaviour"),
    ("BEHAVIOURAL_INCIDENT_STUDENT_INVOLVEMENTS", "temp_fact_Behaviour"),
    ("REGISTRATION_FORMS", "temp_dim_RegGroup"),
    ("STUDENT_REGISTRATION_FORM_MEMBERSHIPS", "temp_dim_RegGroupMembership"),
    ("STUDENT_SEN_NEEDS", "dim_SENDNeed"),
    ("EXAM_RESULTS", "fact_Attainment"),
    ("SUMMATIVE_ASSESSMENT_MARKS", "temp_sum"),
    ("STANDARDISED_ASSESSMENT_MARKS", "temp_stan"),
    ("STAFF", "dim_Staff"),
    ("STAFF_ABSENCES", "fact_StaffAbsence"),
    ("LESSON_ATTENDANCE", "fact_AttendanceLesson"),
    ("COURSES", "temp_Courses"),
    ("COURSE_LEADS", "temp_CourseLeads"),
    ("COURSE_ENROLMENTS", "fact_CourseEnrolment"),
    ("STAFF_CONTRACT", "dim_StaffContractual"),
    ("STUDENT_SCHOOL_ENROLMENTS", "temp_LeavingReason"),
    ("ACADEMIC_YEARS", "dim_AcademicYear"),
    ("USER_DEFINED_FIELDS", "temp_udf"),
    ("USER_DEFINED_FIELDS_VALUES", "dim_UserDefinedFields"), 
    ("BEHAVIOURAL_INCIDENT_ACTIONS", "temp_BActions"),
    ("DETENTIONS", "fact_Detention"),
    ("INTERNAL_EXCLUSIONS", "fact_InternalExclusion"),
    ("POINT_AWARDS", "fact_PointAward"),
]